**Generated By Gemini Pro, Edited, Debugged, Compiled and Prompted By Manim Community Nepal**

In [ ]:
from manim import *
import os

class VAEEnhancedFlow(MovingCameraScene):
    def construct(self):
        # ==========================================
        # 1. STYLES & CONFIG
        # ==========================================
        self.camera.background_color = "#111111"
        
        # Color Palette
        C_TRAIN   = "#00FF7F"    # Green
        C_DATA    = "#FFFFFF"    # White
        C_LATENT  = "#FFD700"    # Gold
        C_NOISE   = "#00BFFF"    # Blue
        C_OP      = "#FF00FF"    # Magenta
        C_WIRE    = GRAY_C
        
        # --- LAYOUT CONTROLS ---
        CODE_SCALE = 0.60
        DIAGRAM_SCALE = 0.75 
        
        # FIX 1: Move diagram significantly Right to clear the Code
        DIAGRAM_SHIFT = RIGHT * 5.0
        
        # FIX 2: Moderate Zoom to fit the separated layout
        FINAL_FRAME_WIDTH = 22.0
        
        # ==========================================
        # 2. COMPONENT FACTORIES
        # ==========================================
        def create_tensor_node(label, sub, pos, color=C_DATA):
            c = Circle(radius=0.40, color=color, fill_color=BLACK, fill_opacity=0.8, stroke_width=2)
            c.move_to(pos)
            l = MathTex(label, font_size=40, color=color).move_to(c)
            s = Text(sub, font_size=20, color=GRAY).next_to(c, DOWN, buff=0.1)
            return VGroup(c, l, s)

        def create_weight_block(label, pos):
            # Compact box
            box = RoundedRectangle(corner_radius=0.1, width=1.5, height=0.85, 
                                 color=C_TRAIN, fill_color="#002200", fill_opacity=0.8)
            box.move_to(pos)
            lbl = MathTex(label, font_size=32, color=C_TRAIN).move_to(box)
            tag = MathTex("W, b", font_size=20, color=C_TRAIN).next_to(box, UP, buff=0.05)
            return VGroup(box, lbl, tag)

        def create_op(symbol, pos):
            c = Circle(radius=0.28, color=C_OP, fill_color=BLACK, fill_opacity=1)
            c.move_to(pos)
            sym = MathTex(symbol, font_size=36, color=C_OP).move_to(c)
            return VGroup(c, sym)

        def create_wire(start, end, label=None):
            l = Line(start, end, color=C_WIRE, stroke_width=2)
            grp = VGroup(l)
            if label:
                lbl = MathTex(label, font_size=26, color=C_DATA)
                lbl.add_background_rectangle(color=BLACK, opacity=1.0, buff=0.05)
                lbl.move_to(l.get_center() + UP*0.2)
                grp.add(lbl)
            return grp

        # ==========================================
        # 3. BUILD THE COMPUTATION GRAPH
        # ==========================================
        
        # FIX 3: COMPACT COORDINATES (Shortened Lines)
        # Relative to Diagram Center
        
        X_IN = -4.5      # Was -5.5
        X_ENC = -2.5     # Was -3.0
        X_SPLIT = -0.5   # Was -0.8
        X_BLOCKS = 1.0   # Block position
        X_LAT = 2.2      # Was 2.5
        
        # Z Calculation Cluster
        X_Z_CALC = 3.8   # Position for noise/ops
        
        # Z Node Position (Result)
        X_Z_NODE = X_Z_CALC + 1.0  # ~4.8
        
        # Decoder - Critical Spacing
        # Z Node edge is at ~5.2. Decoder box edge starts at X_DEC - 0.75.
        # Placing X_DEC at 6.8 gives start at 6.05 -> ~0.8 unit gap.
        X_DEC = 6.8     
        X_OUT = 8.5      # Compacted output
        
        Y_MAIN = 0
        Y_UP = 2.4   
        Y_DOWN = -2.4
        
        # --- NODES ---
        n_x = create_tensor_node("x", "Input", [X_IN, Y_MAIN, 0])
        b_enc = create_weight_block("W_{enc}", [X_ENC, Y_MAIN, 0])
        n_h = create_tensor_node("h", "Hidden", [X_SPLIT, Y_MAIN, 0])
        
        b_mu = create_weight_block("W_{\mu}", [X_BLOCKS, Y_UP, 0])
        b_var = create_weight_block("W_{\sigma}", [X_BLOCKS, Y_DOWN, 0])
        
        n_mu = create_tensor_node(r"\mu", "Mean", [X_LAT, Y_UP, 0], C_LATENT)
        n_logvar = create_tensor_node(r"\ln\sigma^2", "LogVar", [X_LAT, Y_DOWN, 0], C_LATENT)
        
        # Noise
        n_eps = create_tensor_node(r"\epsilon", r"\mathcal{N}", [X_Z_CALC, Y_DOWN - 1.5, 0], C_NOISE)
        
        # Ops
        op_std = create_op(r"e^{\frac{\cdot}{2}}", [X_LAT+0.9, Y_DOWN, 0]) 
        op_mul = create_op(r"\times", [X_Z_CALC, Y_DOWN, 0])
        op_add = create_op("+", [X_Z_CALC, Y_UP, 0])
        
        n_z = create_tensor_node("z", "Latent", [X_Z_NODE, Y_MAIN, 0], C_LATENT)
        b_dec = create_weight_block("W_{dec}", [X_DEC, Y_MAIN, 0])
        n_out = create_tensor_node(r"\hat{x}", "Recon", [X_OUT, Y_MAIN, 0])

        # --- WIRES ---
        wires = VGroup()
        wires.add(create_wire(n_x.get_right(), b_enc.get_left()))
        wires.add(create_wire(b_enc.get_right(), n_h.get_left()))
        
        # Elbows
        w_h_mu_1 = Line(n_h.get_right(), [b_mu.get_left()[0]-0.4, 0, 0], color=C_WIRE)
        w_h_mu_2 = Line([b_mu.get_left()[0]-0.4, 0, 0], [b_mu.get_left()[0]-0.4, Y_UP, 0], color=C_WIRE)
        w_h_mu_3 = Line([b_mu.get_left()[0]-0.4, Y_UP, 0], b_mu.get_left(), color=C_WIRE)
        wires.add(VGroup(w_h_mu_1, w_h_mu_2, w_h_mu_3))
        
        w_h_var_2 = Line([b_mu.get_left()[0]-0.4, 0, 0], [b_var.get_left()[0]-0.4, Y_DOWN, 0], color=C_WIRE)
        w_h_var_3 = Line([b_var.get_left()[0]-0.4, Y_DOWN, 0], b_var.get_left(), color=C_WIRE)
        wires.add(VGroup(w_h_var_2, w_h_var_3))

        wires.add(create_wire(b_mu.get_right(), n_mu.get_left()))
        wires.add(create_wire(b_var.get_right(), n_logvar.get_left()))
        wires.add(create_wire(n_logvar.get_right(), op_std.get_left()))
        wires.add(create_wire(op_std.get_right(), op_mul.get_left(), r"\sigma"))
        wires.add(create_wire(n_eps.get_top(), op_mul.get_bottom()))
        wires.add(create_wire(op_mul.get_top(), op_add.get_bottom(), r"\epsilon \cdot \sigma"))
        wires.add(create_wire(n_mu.get_right(), op_add.get_left()))
        
        w_add_z_1 = Line(op_add.get_right(), [n_z.get_left()[0]-0.3, Y_UP, 0], color=C_WIRE)
        w_add_z_2 = Line([n_z.get_left()[0]-0.3, Y_UP, 0], [n_z.get_left()[0]-0.3, 0, 0], color=C_WIRE)
        w_add_z_3 = Line([n_z.get_left()[0]-0.3, 0, 0], n_z.get_left(), color=C_WIRE)
        wires.add(VGroup(w_add_z_1, w_add_z_2, w_add_z_3))
        
        wires.add(create_wire(n_z.get_right(), b_dec.get_left()))
        wires.add(create_wire(b_dec.get_right(), n_out.get_left()))

        diagram = VGroup(wires, n_x, b_enc, n_h, b_mu, b_var, n_mu, n_logvar, 
                         op_std, n_eps, op_mul, op_add, n_z, b_dec, n_out)

        # ==========================================
        # 4. PYTHON CODE
        # ==========================================
        code_str = """class VAE(nn.Module):
    def __init__(self):
        # Trainable Layers
        self.enc = nn.Linear(D_in, D_h)
        self.w_mu = nn.Linear(D_h, D_z)
        self.w_var = nn.Linear(D_h, D_z)
        self.dec = nn.Linear(D_z, D_in)

    def reparameterize(self, mu, logvar):
        # The Trick: z = mu + eps * sigma
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        h = torch.relu(self.enc(x))
        mu, logvar = self.w_mu(h), self.w_var(h)
        z = self.reparameterize(mu, logvar)
        return self.dec(z)"""

        code_filename = "vae_clean.py"
        with open(code_filename, "w") as f: f.write(code_str)
        
        code_obj = Code(code_filename, language="python").scale(CODE_SCALE)
        code_obj.to_edge(LEFT, buff=0.5)
        code_lines = code_obj[2]

        # ==========================================
        # 5. ANIMATION SEQUENCE
        # ==========================================
        
        # Move diagram
        diagram.scale(DIAGRAM_SCALE).move_to(DIAGRAM_SHIFT)
        
        # Center camera between Code (Left) and Diagram (Right)
        # Diagram is at +5.0, Code is at ~ -6.0. Center is ~ -0.5
        center_point = (code_obj.get_center() + diagram.get_center()) / 2
        
        self.camera.frame.move_to(center_point).set(width=FINAL_FRAME_WIDTH)

        title = Text("VAE Computation Graph", font_size=56)
        title.move_to(self.camera.frame.get_top() + DOWN * 2.0)
        
        desc_text = Text("Initialization...", font_size=32, color=GRAY)
        desc_text.move_to(self.camera.frame.get_bottom() + UP * 2.5)
        
        def update_desc(new_text):
            nonlocal desc_text
            t = Text(new_text, font_size=32, color=WHITE).move_to(desc_text.get_center())
            self.play(Transform(desc_text, t), run_time=0.5)

        self.play(Write(title), FadeIn(code_obj))
        self.play(FadeIn(diagram), Write(desc_text))
        self.wait(1)

        def flow_pulse(mob_path, color=C_TRAIN, run_time=1.0):
            if isinstance(mob_path, VGroup):
                lines = [sub for sub in mob_path if isinstance(sub, Line)]
                if not lines:
                    path = mob_path.copy()
                else:
                    pts = [l.get_start() for l in lines]
                    pts.append(lines[-1].get_end())
                    path = VMobject().set_points_as_corners(pts)
            elif isinstance(mob_path, Line):
                path = mob_path.copy()
            else:
                path = mob_path.copy()
            path.set_stroke(color=color, width=8, opacity=1)
            self.play(ShowPassingFlash(path, time_width=0.3, run_time=run_time))

        # 1. INIT
        update_desc("Defining Learnable Weights (Green)")
        hl = SurroundingRectangle(code_lines[2:7], color=C_TRAIN, fill_opacity=0.1)
        self.play(Create(hl))
        
        trainables = [b_enc, b_mu, b_var, b_dec]
        self.play(
            *[Indicate(b[0], color=C_TRAIN, scale_factor=1.1) for b in trainables],
            run_time=1.5
        )
        self.wait(0.5)

        # 2. ENCODER
        update_desc("Input x flows to hidden state h")
        self.play(Transform(hl, SurroundingRectangle(code_lines[15], color=C_OP)))
        
        self.play(Indicate(n_x, color=WHITE))
        flow_pulse(wires[0], C_DATA) 
        self.play(Indicate(b_enc[0], color=C_TRAIN))
        flow_pulse(wires[1], C_DATA)
        self.play(Indicate(n_h, color=WHITE))

        # 3. SPLIT
        update_desc("Compute Mean (mu) & Log Variance")
        self.play(Transform(hl, SurroundingRectangle(code_lines[16], color=C_OP)))
        
        self.play(
            Indicate(b_mu[0], color=C_TRAIN), 
            Indicate(b_var[0], color=C_TRAIN)
        )
        self.play(
            Indicate(n_mu, color=C_LATENT), 
            Indicate(n_logvar, color=C_LATENT)
        )

        # 4. REPARAMETERIZE
        update_desc("Reparameterize: z = mu + eps * sigma")
        self.play(Transform(hl, SurroundingRectangle(code_lines[9:13], color=C_NOISE)))
        
        focus_group = VGroup(n_mu, n_logvar, n_eps, n_z)
        self.play(
            self.camera.frame.animate.move_to(focus_group.get_center()).set(width=12.0),
            run_time=1.5
        )

        self.play(Indicate(op_std, color=C_OP))
        flow_pulse(wires[6], C_LATENT) 
        self.play(Flash(n_eps, color=C_NOISE, flash_radius=0.5))
        self.play(Indicate(n_eps, color=C_NOISE))
        flow_pulse(wires[7], C_LATENT) 
        flow_pulse(wires[8], C_NOISE)  
        self.play(Indicate(op_mul, color=C_OP))
        flow_pulse(wires[9], C_NOISE)  
        flow_pulse(wires[10], C_LATENT)
        self.play(Indicate(op_add, color=C_OP))
        flow_pulse(wires[11], C_LATENT)
        self.play(Indicate(n_z, color=C_LATENT, scale_factor=1.2))

        # 5. DECODE
        update_desc("Reconstruct x from z")
        # Zoom back to full view
        self.play(
            self.camera.frame.animate.move_to(center_point).set(width=FINAL_FRAME_WIDTH),
            Transform(hl, SurroundingRectangle(code_lines[18], color=C_OP)),
            run_time=1.5
        )
        
        flow_pulse(wires[12], C_LATENT) 
        self.play(Indicate(b_dec[0], color=C_TRAIN))
        flow_pulse(wires[13], C_DATA)   
        self.play(Indicate(n_out, color=WHITE, scale_factor=1.2))

        self.wait(2)
        if os.path.exists(code_filename): os.remove(code_filename)


%manim -qk -v warning VAEEnhancedFlow